<a href="https://colab.research.google.com/github/hojuelita/text_classification/blob/main/Ejemplo_vectorizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métodos de embedding

In [3]:

!pip install gensim
# recuperamos el texto desde el git
import requests
import pandas as pd
import re # Added import for regular expressions

import gensim
from gensim.models import Word2Vec
import nltk


url_texto = "https://raw.githubusercontent.com/hojuelita/text_classification/refs/heads/main/data/Cuentos/A_una_bogotana.txt"
response = requests.get(url_texto)
texto = response.text

print("Original text:")

def clean_text(text, minusculas=True):
    # Remove everything inside brackets [...]
    text = re.sub(r"\[.*\]", '', text)
    # Remove everything inside parenthesis (...)
    text = re.sub(r'\(.*\)', '', text)
    # Remove continuous asterisks *
    text = re.sub(r'\*+', '', text)
    # Optional: remove leading/trailing whitespace that might result from cleaning
    if(minusculas):
      text = text.lower()
    text = text.strip()
    return text

def split_paragraphs(text):
    """
    Split the text into paragraphs based on double newlines.
    """
    paragraphs = text.split('\n\n')
    splitted = {}
    for j,par in enumerate(paragraphs):
      splitted[j] = re.sub("\n", "", par.strip())
    return splitted

cleaned_texto = clean_text(texto, minusculas=True)
print("\nCleaned text:")
print(cleaned_texto)

Original text:

Cleaned text:
a una bogotana 

el pasillo, señora, hermosa niña, es como un lento y rosado vals. vea
usted cómo aquellos dos enamorados pueden llevar el compás, en medio de
la más ardiente conversación. el dice que los lindos ojos de una mujer
valen por todos los astros, y los lindos labios por todas las rosas.
como ella quiere demostrar lo contrario, le mira con los bellísimos ojos
suyos, le sonríe con sus inefables labios, que son en un todo iguales a
aquellos con que la señorita de abril dió el primer beso al caballero de
mayo. el pasillo, señora, hermosa niña, es como un lento y rosado vals.

¡oh, sí, sí! la fuerza de una pasión es mayor, infinitas veces, que el
empuje de ese enorme y poderoso tequendama. ¿usted conoce la catarata?

dicen que sus aguas saltan de un clima a otro. que allá abajo hay palmas
y flores; que arriba, en la roca que conoció la espada de bolívar, hace
frío. ¡qué delicia estar allá abajo, señora, dos que se quieren! la
soberana armonía de la n

In [4]:
# vamos a limpiar el texto quitando caraceres especiales,
# todo lo que esté entre []
parrafos = split_paragraphs(cleaned_texto)
parrafos

{0: 'a una bogotana',
 1: 'el pasillo, señora, hermosa niña, es como un lento y rosado vals. veausted cómo aquellos dos enamorados pueden llevar el compás, en medio dela más ardiente conversación. el dice que los lindos ojos de una mujervalen por todos los astros, y los lindos labios por todas las rosas.como ella quiere demostrar lo contrario, le mira con los bellísimos ojossuyos, le sonríe con sus inefables labios, que son en un todo iguales aaquellos con que la señorita de abril dió el primer beso al caballero demayo. el pasillo, señora, hermosa niña, es como un lento y rosado vals.',
 2: '¡oh, sí, sí! la fuerza de una pasión es mayor, infinitas veces, que elempuje de ese enorme y poderoso tequendama. ¿usted conoce la catarata?',
 3: 'dicen que sus aguas saltan de un clima a otro. que allá abajo hay palmasy flores; que arriba, en la roca que conoció la espada de bolívar, hacefrío. ¡qué delicia estar allá abajo, señora, dos que se quieren! lasoberana armonía de la naturaleza pondría u

Una vez que tengamos los párrafos ya separados podemos vectorizarlos, lo vamos a hacer primero con word2vec pero también vamos a probarlo con FastText y Roberta

## Word2Vec

In [5]:
!pip install gensim

In [6]:
# modelo para oraciones de nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# revisar el modelo usado por spacy

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [9]:
nltk.sent_tokenize(parrafo)[0].split()

NameError: name 'parrafo' is not defined

In [10]:
# tokenizacion en palabras
corpus = []
corpus_oraciones = []
for parrafo in parrafos.values():
  print(parrafo)
  try:
    corpus_oraciones.append( nltk.sent_tokenize(parrafo)[0].split() )
  except:
    print(parrafo)
  oraciones = [palabra for oracion in nltk.sent_tokenize(parrafo) for palabra in oracion.split()]
  corpus += oraciones
  print(corpus)

a una bogotana
['a', 'una', 'bogotana']
el pasillo, señora, hermosa niña, es como un lento y rosado vals. veausted cómo aquellos dos enamorados pueden llevar el compás, en medio dela más ardiente conversación. el dice que los lindos ojos de una mujervalen por todos los astros, y los lindos labios por todas las rosas.como ella quiere demostrar lo contrario, le mira con los bellísimos ojossuyos, le sonríe con sus inefables labios, que son en un todo iguales aaquellos con que la señorita de abril dió el primer beso al caballero demayo. el pasillo, señora, hermosa niña, es como un lento y rosado vals.
['a', 'una', 'bogotana', 'el', 'pasillo,', 'señora,', 'hermosa', 'niña,', 'es', 'como', 'un', 'lento', 'y', 'rosado', 'vals.', 'veausted', 'cómo', 'aquellos', 'dos', 'enamorados', 'pueden', 'llevar', 'el', 'compás,', 'en', 'medio', 'dela', 'más', 'ardiente', 'conversación.', 'el', 'dice', 'que', 'los', 'lindos', 'ojos', 'de', 'una', 'mujervalen', 'por', 'todos', 'los', 'astros,', 'y', 'los', 

In [11]:
len(corpus_oraciones)

7

In [12]:
# instanciamos el modelo
# usando las oraciones
modelo = Word2Vec(sentences=corpus_oraciones, vector_size=100, window=5, min_count=1, workers=4)


In [13]:
modelo.wv.key_to_index

{'de': 0,
 'un': 1,
 'la': 2,
 'en': 3,
 'y': 4,
 'una': 5,
 'a': 6,
 'las': 7,
 'que': 8,
 'señora,': 9,
 'va?': 10,
 '¿se': 11,
 'conque': 12,
 'adelfa.': 13,
 'selene': 14,
 'pálida': 15,
 'enfermiza': 16,
 'clavel,y': 17,
 'sirio': 18,
 'rosa,': 19,
 'cortaré': 20,
 'venus': 21,
 'estrella': 22,
 'por': 23,
 'pasar': 24,
 'paraíso.al': 25,
 'del': 26,
 'vírgenes': 27,
 'cuidan': 28,
 'jardines': 29,
 'los': 30,
 'lirio': 31,
 'cortar': 32,
 'láctea,a': 33,
 'vía': 34,
 'implacables!--iré,': 35,
 'flechas': 36,
 'hondos': 37,
 'dolores': 38,
 '¡ay,por': 39,
 'heridas,': 40,
 'permitan,': 41,
 'lo': 42,
 'me': 43,
 'pegaso': 44,
 'mi': 45,
 'alas': 46,
 'heridas': 47,
 'cuanto': 48,
 'otro.': 49,
 'clima': 50,
 'saltan': 51,
 'aguas': 52,
 'sus': 53,
 'dicen': 54,
 'sí!': 55,
 'sí,': 56,
 '¡oh,': 57,
 'vals.': 58,
 'rosado': 59,
 'lento': 60,
 'como': 61,
 'es': 62,
 'niña,': 63,
 'hermosa': 64,
 'pasillo,': 65,
 'el': 66,
 'bogotana': 67}

In [14]:
modelo.wv.get_vector('bogotana')

array([ 8.5969213e-03, -5.8590295e-03, -7.6059494e-03, -1.4025867e-03,
        2.9744648e-03, -1.9831157e-03,  5.8881771e-03, -8.4122792e-03,
       -9.0060961e-03,  7.5791953e-03,  6.3356413e-03, -4.0856195e-03,
       -4.0202928e-03, -3.5549463e-03, -5.1817893e-05,  9.0750484e-03,
        9.5188618e-04, -2.0490445e-03, -3.4158814e-03, -2.0868706e-03,
       -9.1953576e-03,  6.8275211e-03, -9.0328362e-03, -3.0905807e-03,
       -7.1195271e-03,  9.0612294e-03,  5.7918443e-03, -4.7887804e-04,
        3.9187074e-03, -2.4324870e-03,  9.8090256e-03, -3.6267508e-03,
        4.8595010e-03, -3.7946510e-03,  9.5509859e-03,  9.4973259e-03,
        6.2314663e-03, -2.4811030e-04, -1.6862452e-03, -8.6564943e-03,
        9.3346573e-03, -8.4423125e-03, -6.5561975e-03,  5.5810092e-03,
       -2.2827517e-03,  7.9754563e-03, -7.8672096e-03,  9.1965441e-03,
        2.2511708e-03, -8.5561238e-03,  3.9443877e-03, -4.9556256e-03,
        6.3843774e-03,  6.5867458e-03, -8.7615130e-03,  6.4632809e-03,
      

In [15]:
modelo.wv.most_similar('bogotana')

[('una', 0.17856454849243164),
 ('lo', 0.1772552728652954),
 ('un', 0.15850168466567993),
 ('enfermiza', 0.13965535163879395),
 ('estrella', 0.13599935173988342),
 ('pálida', 0.12827648222446442),
 ('alas', 0.12635980546474457),
 ('a', 0.10168150067329407),
 ('sus', 0.09211434423923492),
 ('selene', 0.09014482796192169)]

# Tarea

Tarea en 3 etapas
1. Poner nb en repo para poder cargarlo desde colab
1. Cargar cada uno de los archivos de texto (novelas y cuentos)
    1.1 Hacer limpieza de texto para quitar ya sea los asteriscos * * * o las anotaciones entre brackets [] y lo que consideres
2. Separar los textos en oraciones ajenas en un objeto en memoria (salvar las oraciones en disco)
3. Entrenar w2v con este dataset. Hacer pruebas de similaridad.
    3.1 Si se ven bien salvar los vectores con FAISS